In [12]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/D-alanine_opt_L-ala_cell-17O_magres.magres')

In [13]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [14]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [15]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [16]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-62.93377238 166.22451312 -80.56157078]
 [172.4697815   65.96113206  24.7350798 ]
 [  8.22591105 -25.69604004 -51.81500928]]

17O2 sigma:
 [[ -53.82014758  218.76780944   55.02898048]
 [ 185.81011053  115.33568175  -88.18842382]
 [  18.64191826  -51.17834508 -172.26916506]]

17O3 sigma:
 [[-62.93377238 166.22451312  80.56157078]
 [172.4697815   65.96113206 -24.7350798 ]
 [ -8.22591105  25.69604004 -51.81500928]]

17O4 sigma:
 [[ -62.93377238 -166.22451312   80.56157078]
 [-172.4697815    65.96113206   24.7350798 ]
 [  -8.22591105  -25.69604004  -51.81500928]]

17O5 sigma:
 [[ -62.93377238 -166.22451312  -80.56157078]
 [-172.4697815    65.96113206  -24.7350798 ]
 [   8.22591105   25.69604004  -51.81500928]]

17O6 sigma:
 [[ -53.82014758  218.76780944  -55.02898048]
 [ 185.81011053  115.33568175   88.18842382]
 [ -18.64191826   51.17834508 -172.26916506]]

17O7 sigma:
 [[ -53.82014758 -218.76780944  -55.02898048]
 [-185.81011053  115.33568175  -88.18842382]
 [ -18.64191826

In [17]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.3985376856896785

17O2 sigma:
 8.257931708019381

17O3 sigma:
 6.398537685689589

17O4 sigma:
 6.398537685689671

17O5 sigma:
 6.398537685689582

17O6 sigma:
 8.25793170801937

17O7 sigma:
 8.257931708019346

17O8 sigma:
 8.257931708019344



In [23]:
Q = -0.0256 #electric quadrupole moment for O17 in barn

CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + l = 1 + l = 2) Tensor from magres
CS_total[0,0] = -60.4933; CS_total[0,1] = 219.1870; CS_total[0,2] = 56.1644;
CS_total[1,0] = 186.1375; CS_total[1,1] = 116.7557; CS_total[1,2] = -88.2667;
CS_total[2,0] = 18.5468; CS_total[2,1] = -51.1340; CS_total[2,2] =  -175.7186;

Cs = np.zeros((3,3)) # CS symmetric (l = 0 + l = 2) 

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)
efg[0,0]= 0.7155; efg[0,1]= -0.1609; efg[0,2]=  0.1466;
efg[1,0]= efg[0,1]; efg[1,1]= 0.6168; efg[1,2]= -0.2660;
efg[2,0]= efg[0,2]; efg[2,1]= efg[1,2]; efg[2,2]= -1.3324;

# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 

V = efg*Q*234.9647

print(V)

print(Cs)

[[-4.30380142  0.967829   -0.88181312]
 [ 0.967829   -3.71011141  1.60001562]
 [-0.88181312  1.60001562  8.01451434]]
[[ -60.4933   202.66225   37.3556 ]
 [ 202.66225  116.7557   -69.70035]
 [  37.3556   -69.70035 -175.7186 ]]


In [24]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [-5.23153997 -3.04131759  8.27345907] 

 Unsorted Eigenvectors:
 [[-0.77121777  0.63377395 -0.05961322]
 [ 0.62383601  0.7711057   0.12737592]
 [-0.12669564 -0.0610457   0.99006143]] 

Sorted Eigenvalues: 
 [-3.04131759 -5.23153997  8.27345907] 

Sorted Eigenvectors: 
 [[ 0.63377395 -0.77121777 -0.05961322]
 [ 0.7711057   0.62383601  0.12737592]
 [-0.0610457  -0.12669564  0.99006143]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 252.75959619 -116.61048552 -255.60531068] 

 Unsorted Eigenvectors:
 [[-0.53335865 -0.60274307 -0.59348913]
 [-0.841054    0.30296462  0.44815244]
 [ 0.09031457 -0.73818238  0.66852827]] 

Sorted Eigenvalues: 
 [-116.61048552 -255.60531068  252.75959619] 

Sorted Eigenvectors: 
 [[-0.60274307 -0.59348913 -0.53335865]
 [ 0.30296462  0.44815244 -0.841054  ]
 [-0.73818238  0.66852827  0.09031457]] 



In [25]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -3.0413175875451417 -5.231539969167204 8.27345906634435
CSA Tensor Components δyy, δxx, δzz: 
 -116.61048551556347 -255.60531067753234 252.7595961930956


In [26]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Value']))

Qauntity           Value
------------  ----------
CQ (MHz)        8.27346
etaq            0.264729
iso_cs (ppm)  -39.8187
csa (ppm)     292.578
etas            0.475069


In [27]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.77121777  0.63377395 -0.05961322]
 [ 0.62383601  0.7711057   0.12737592]
 [-0.12669564 -0.0610457   0.99006143]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
25.726140060598578 8.084625330234006 64.91992914687177 

Direction cosine csa: 

[[-0.59348913 -0.60274307 -0.53335865]
 [ 0.44815244  0.30296462 -0.841054  ]
 [ 0.66852827 -0.73818238  0.09031457]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
-47.834719383303835 84.81829542617807 -57.61896616390186 



In [28]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -41.02506590090356 chi: 89.19312408771229 xi: -82.83085484945434 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)


CSA Tensor in Crystal Frame: 
 [[90.04399411  0.1909     -1.44775   ]
 [ 0.1909     83.75632904  4.55995   ]
 [-1.44775     4.55995    78.09956279]]
CSA Tensor in Tenon Frame: 
 [[82.75286415  2.60154732 -4.34538642]
 [ 2.60154732 86.52299334  5.29813785]
 [-4.34538642  5.29813785 82.62402845]]
Quad Tensor in Crystal Frame: 
 [[-0.27181069  1.76629262  1.06053855]
 [ 1.76629262 -0.05626958  0.25368998]
 [ 1.06053855  0.25368998  0.32808027]]
Quad Tensor in Tenon Frame: 
 [[ 0.2801114  -0.83813975 -1.35455866]
 [-0.83813975 -1.36878474 -0.54341786]
 [-1.35455866 -0.54341786  1.08867334]]
